# 05 - Final evaluation on the held-out test set

This notebook evaluates the Linear SVM, Random Forest and CNN using the official test split. Model selection is based on validation macro-F1 from notebooks 03 and 04. Test metrics are reported once as the final generalisation estimate and must not be used for further model tuning.

**Run notebooks 01-04 first.** Notebooks 03 and 04 must be rerun after the latest corrections so that all three saved models use the same 12 classes, including `silence`.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import ConfusionMatrixDisplay

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import evaluate_classifier
from src.deep_learning import SmallAudioCNN, SpeechCommandsMelDataset, evaluate_cnn

DATA_DIR = PROJECT_ROOT / 'notebooks' / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'
INDEX_PATH = DATA_DIR / 'audio_index_with_silence.csv'
FEATURE_PATH = DATA_DIR / 'mfcc_features.npz'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

for required_path in [INDEX_PATH, FEATURE_PATH, MODEL_DIR / 'linear_svm.joblib',
                      MODEL_DIR / 'random_forest.joblib', MODEL_DIR / 'small_audio_cnn.pt',
                      MODEL_DIR / 'validation_model_comparison.json',
                      MODEL_DIR / 'cnn_validation_model_comparison.json']:
    if not required_path.is_file():
        raise FileNotFoundError(f'Missing {required_path}. Run corrected notebooks 02-04 first.')

print('Project root:', PROJECT_ROOT)
print('Evaluation device:', DEVICE)

## 2. Load the untouched test split and verify class alignment

In [ ]:
audio_index = pd.read_csv(INDEX_PATH)
test_frame = audio_index[audio_index['split'].astype(str) == 'test'].copy()
with np.load(FEATURE_PATH, allow_pickle=True) as data:
    X_test = data['X_test'].astype(np.float32)
    y_test = np.asarray(data['y_test'], dtype=str)

index_classes = sorted(audio_index['target'].astype(str).unique().tolist())
feature_classes = sorted(np.unique(y_test).tolist())
assert index_classes == feature_classes, (
    f'Class mismatch between audio index and MFCC test features: {index_classes} vs {feature_classes}'
)
assert 'silence' in index_classes and len(index_classes) == 12
print('Test audio clips:', len(test_frame))
print('MFCC test matrix:', X_test.shape)
print('Classes:', index_classes)
display(test_frame['target'].value_counts().sort_index().to_frame('test_examples'))

## 3. Load validation results and identify the validation-selected model

In [ ]:
classical_validation = json.loads((MODEL_DIR / 'validation_model_comparison.json').read_text(encoding='utf-8'))
cnn_validation = json.loads((MODEL_DIR / 'cnn_validation_model_comparison.json').read_text(encoding='utf-8'))
validation_rows = list(classical_validation)
validation_rows.append({'model': 'small_audio_cnn', **cnn_validation})
validation_comparison = pd.DataFrame(validation_rows).sort_values(
    ['macro_f1', 'balanced_accuracy'], ascending=False
).reset_index(drop=True)
display(validation_comparison.round(4))
validation_winner = str(validation_comparison.iloc[0]['model'])
print('Model selected using validation macro-F1:', validation_winner)
print('This selection is fixed before examining test metrics.')

## 4. Evaluate Linear SVM and Random Forest

In [ ]:
test_results = {}
test_rows = []
for model_name in ['linear_svm', 'random_forest']:
    bundle = joblib.load(MODEL_DIR / f'{model_name}.joblib')
    result = evaluate_classifier(bundle['model'], X_test, y_test)
    test_results[model_name] = result
    test_rows.append({'model': model_name, **result['metrics']})
    print(f'\\n{model_name} per-class test report')
    display(pd.DataFrame(result['report']).T.round(3))
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(
        confusion_matrix=result['confusion_matrix'], display_labels=result['labels']
    ).plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
    ax.set_title(f'{model_name.replace("_", " ").title()} - test confusion matrix')
    plt.tight_layout(); plt.show()

## 5. Evaluate the CNN on the same test recordings

In [ ]:
checkpoint = torch.load(MODEL_DIR / 'small_audio_cnn.pt', map_location=DEVICE)
cnn_labels = list(checkpoint['labels'])
assert cnn_labels == index_classes, f'CNN classes do not match the test index: {cnn_labels}'
label_to_id = {label: i for i, label in enumerate(cnn_labels)}
test_dataset = SpeechCommandsMelDataset(test_frame, label_to_id, sample_rate=16_000,
                                        n_mels=int(checkpoint['n_mels']))
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
cnn_model = SmallAudioCNN(n_classes=len(cnn_labels)).to(DEVICE)
cnn_model.load_state_dict(checkpoint['state_dict'])
cnn_result = evaluate_cnn(cnn_model, test_loader, cnn_labels, device=DEVICE)
cnn_metrics = {key: cnn_result[key] for key in [
    'accuracy', 'balanced_accuracy', 'macro_precision', 'macro_recall', 'macro_f1'
]}
test_results['small_audio_cnn'] = {
    'metrics': cnn_metrics,
    'report': cnn_result['report'],
    'confusion_matrix': cnn_result['confusion_matrix'],
    'labels': cnn_labels,
}
test_rows.append({'model': 'small_audio_cnn', **cnn_metrics})
display(pd.DataFrame(cnn_result['report']).T.round(3))
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(confusion_matrix=cnn_result['confusion_matrix'],
                       display_labels=cnn_labels).plot(
    ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False
)
ax.set_title('CNN - test confusion matrix')
plt.tight_layout(); plt.show()

## 6. Compare final test metrics and save the evaluation

In [ ]:
test_comparison = pd.DataFrame(test_rows).set_index('model')
display(test_comparison.sort_values('macro_f1', ascending=False).round(4))

# Keep the validation-selected model as the official candidate. Do not replace it
# just because another model happens to score higher on this test sample.
print('Validation-selected model:', validation_winner)
print('Its held-out test metrics:')
display(test_comparison.loc[[validation_winner]].round(4))

output_path = MODEL_DIR / 'final_test_metrics.json'
output_path.write_text(json.dumps({
    'validation_selected_model': validation_winner,
    'validation_comparison': validation_rows,
    'test_metrics': test_comparison.reset_index().to_dict(orient='records'),
}, indent=2), encoding='utf-8')
for model_name, result in test_results.items():
    pd.DataFrame(result['report']).T.to_csv(MODEL_DIR / f'{model_name}_test_classification_report.csv')
print('Saved final test metrics:', output_path)
print('Final test evaluation complete. Keep these results fixed for the report.')

## Interpretation guide

- **Accuracy** measures the proportion of all predictions that are correct. Because `unknown` is much more common than the command classes, accuracy alone can be misleading.
- **Balanced accuracy** gives each class equal weight through its recall, making it useful for this imbalanced dataset.
- **Macro precision, recall and F1** treat all 12 classes equally. Macro-F1 is the primary comparison metric because it balances precision and recall across common and rare classes.
- **Weighted F1** weights each class by its number of test examples, so it reflects the original class distribution more strongly.
- A confusion matrix shows which spoken commands are being mistaken for other commands. Pay particular attention to errors between similar-sounding words and between `silence` and `unknown`.

The validation-selected model remains the chosen candidate. Test results are a final estimate, not a new tuning signal.